# 🚢 Titanic Survival Predictor

**Goal:** predict whether a passenger survived the Titanic disaster (1912) from details like their age, sex, ticket class and fare.

In the house price project you predicted a **number** (regression). Here you predict a **category** — *survived* or *didn't* — which is called **classification**.

**New skills in this project**
- Classification models (logistic regression, random forest, gradient boosting)
- Accuracy vs **precision** vs **recall** — and why accuracy alone can fool you
- **Confusion matrices**
- Feature engineering from messy text (pulling "Mr", "Mrs", "Master" out of names)
- Making a Kaggle submission and getting on the leaderboard

⏱️ One weekend. 
**How to run:** open [Google Colab](https://colab.research.google.com) → **File → Upload notebook** → choose this file. Run cells top to bottom with **Shift + Enter**.

## 1. Load the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
titanic = pd.read_csv(URL)
print(titanic.shape)
titanic.head()

| Column | Meaning |
|---|---|
| `Survived` | **Target:** 1 = survived, 0 = died |
| `Pclass` | Ticket class: 1 = first, 2 = second, 3 = third |
| `Name`, `Sex`, `Age` | Who they were |
| `SibSp` | Number of siblings/spouses aboard |
| `Parch` | Number of parents/children aboard |
| `Ticket`, `Fare` | Ticket number and price paid |
| `Cabin` | Cabin number (mostly missing) |
| `Embarked` | Port they boarded at: C = Cherbourg, Q = Queenstown, S = Southampton |

In [ ]:
titanic.info()

👀 `Age` has missing values, `Cabin` is mostly missing, and `Embarked` is missing 2 values.

## 2. Explore — who survived?

In [ ]:
print(f"Overall survival rate: {titanic['Survived'].mean():.0%}")
titanic.groupby("Sex")["Survived"].mean()

In [ ]:
titanic.groupby("Pclass")["Survived"].mean()

"Women and children first" — and first-class passengers did much better. Let's see age:

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
titanic[titanic["Survived"] == 1]["Age"].plot(kind="hist", bins=30, alpha=0.6, label="survived", ax=ax)
titanic[titanic["Survived"] == 0]["Age"].plot(kind="hist", bins=30, alpha=0.6, label="died", ax=ax)
ax.set_xlabel("Age"); ax.legend(); ax.set_title("Age of survivors vs non-survivors")
plt.show()

In [ ]:
pd.crosstab([titanic["Sex"], titanic["Pclass"]], titanic["Survived"], normalize="index").round(2)

Almost all first- and second-class women survived; third-class men had very poor odds. A model should find these patterns easily.

## 3. Feature engineering

### Titles from names
Names look like `"Braund, Mr. Owen Harris"`. The **title** (Mr, Mrs, Miss, Master) tells us about sex, age *and* social status — and "Master" means a young boy!

In [ ]:
titanic["Title"] = titanic["Name"].str.extract(r",\s*([^\.]+)\.")
titanic["Title"].value_counts()

In [ ]:
# Group the rare titles together
def clean_title(t):
    if t in ["Mr", "Mrs", "Miss", "Master"]:
        return t
    if t in ["Mlle", "Ms"]:
        return "Miss"
    if t == "Mme":
        return "Mrs"
    return "Rare"

titanic["Title"] = titanic["Title"].apply(clean_title)
titanic.groupby("Title")["Survived"].agg(["mean", "count"]).round(2)

### Family size and other features

In [ ]:
def add_features(df):
    df = df.copy()
    df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.")[0].apply(clean_title)
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
    df["HasCabin"] = df["Cabin"].notna().astype(int)
    return df

data = add_features(titanic)
data.groupby("FamilySize")["Survived"].mean().plot(kind="bar", title="Survival rate by family size")
plt.show()

Travelling alone *or* in a very large family was bad news; small families did best.

## 4. Train / test split

In [ ]:
from sklearn.model_selection import train_test_split

features = ["Pclass", "Sex", "Age", "Fare", "Embarked", "Title", "FamilySize", "IsAlone", "HasCabin"]
X = data[features]
y = data["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(len(X_train), "train /", len(X_test), "test")

`stratify=y` keeps the same survived/died ratio in both sets — important for classification when classes aren't 50/50.

## 5. Preparation pipeline (same idea as the house price project)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric = ["Pclass", "Age", "Fare", "FamilySize", "IsAlone", "HasCabin"]
categorical = ["Sex", "Embarked", "Title"]

preprocessing = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric),
    (make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), categorical),
)

## 6. Baseline

The dumbest classifier: **predict "died" for everyone** (the most common outcome).

In [ ]:
from sklearn.metrics import accuracy_score

baseline_acc = accuracy_score(y_test, np.zeros(len(y_test)))
print(f"Baseline accuracy (everyone dies): {baseline_acc:.1%}")

That's ~62% accurate while being completely useless — it never predicts a single survivor. **This is why accuracy alone can fool you.** Hold that thought.

## 7. Logistic Regression — your first classifier

Despite the name, **logistic regression** is a classification model. It works like linear regression but squashes the output into a **probability between 0 and 1** using an S-shaped curve. If the probability is above 0.5, we predict "survived".

In [ ]:
from sklearn.linear_model import LogisticRegression

log_model = make_pipeline(preprocessing, LogisticRegression(max_iter=1000))
log_model.fit(X_train, y_train)
pred = log_model.predict(X_test)
print(f"Logistic regression accuracy: {accuracy_score(y_test, pred):.1%}")

## 8. The confusion matrix

Accuracy is one number. A **confusion matrix** shows *what kind* of mistakes the model makes:

|  | Predicted died | Predicted survived |
|---|---|---|
| **Actually died** | ✅ True Negative | ❌ False Positive |
| **Actually survived** | ❌ False Negative | ✅ True Positive |

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["died", "survived"], cmap="Blues")
plt.title("Logistic regression")
plt.show()

## 9. Precision and recall

- **Precision** = of the people we *said* survived, how many really did? *"When I say yes, am I right?"*
- **Recall** = of the people who *really* survived, how many did we find? *"Did I catch them all?"*
- **F1 score** = a balance of the two.

Which matters more depends on the problem:
- Spam filter → high **precision** (don't bin real emails).
- Cancer screening → high **recall** (don't miss a sick patient).

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, pred, target_names=["died", "survived"]))

### The threshold trade-off
The model outputs a probability. Changing the cut-off from 0.5 moves the balance between precision and recall:

In [ ]:
from sklearn.metrics import precision_score, recall_score

proba = log_model.predict_proba(X_test)[:, 1]   # probability of "survived"
thresholds = np.arange(0.1, 0.91, 0.05)
precisions = [precision_score(y_test, proba >= t, zero_division=0) for t in thresholds]
recalls = [recall_score(y_test, proba >= t) for t in thresholds]

plt.figure(figsize=(8, 5))
plt.plot(thresholds, precisions, marker="o", label="precision")
plt.plot(thresholds, recalls, marker="o", label="recall")
plt.xlabel("threshold"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Raising the threshold: more precise, but catches fewer survivors")
plt.show()

## 10. Try stronger models with cross-validation

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

models = {
    "Logistic regression": LogisticRegression(max_iter=1000),
    "Random forest": RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42),
    "Gradient boosting": HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=42),
}
scores = {}
for name, m in models.items():
    s = cross_val_score(make_pipeline(preprocessing, m), X_train, y_train, cv=5, scoring="accuracy")
    scores[name] = s.mean()
    print(f"{name:22s} {s.mean():.1%} (± {s.std():.1%})")

## 11. Final check on the test set

In [ ]:
best_name = max(scores, key=scores.get)
final_model = make_pipeline(preprocessing, models[best_name]).fit(X_train, y_train)
final_pred = final_model.predict(X_test)
print("Chosen model:", best_name)
print(f"Test accuracy: {accuracy_score(y_test, final_pred):.1%}  (baseline was {baseline_acc:.1%})")
print(classification_report(y_test, final_pred, target_names=["died", "survived"]))
ConfusionMatrixDisplay.from_predictions(y_test, final_pred, display_labels=["died", "survived"], cmap="Greens")
plt.show()

## 12. Would *you* have survived?

In [ ]:
me = pd.DataFrame([{
    "Name": "Tulshibagwale, Mr. Sumant",   # change the title to Mrs/Miss/Master to experiment
    "Pclass": 3, "Sex": "male", "Age": 25, "SibSp": 0, "Parch": 0,
    "Fare": 8.0, "Embarked": "S", "Cabin": None,
}])
me = add_features(me)
p = final_model.predict_proba(me[features])[0, 1]
print(f"Chance of survival: {p:.0%}")

## 13. 🏆 Submit to Kaggle

The data above is Kaggle's **training** file. To get on the leaderboard:

1. Create a free account at [kaggle.com](https://www.kaggle.com) and join the [Titanic competition](https://www.kaggle.com/competitions/titanic).
2. Go to the **Data** tab and download `train.csv` and `test.csv`.
3. In Colab, click the 📁 folder icon on the left and drag both files in.
4. Run the cell below — it trains on *all* of `train.csv` and writes `submission.csv`.
5. Download `submission.csv` and upload it on Kaggle via **Submit Prediction**.

A score around **77–79%** is solid for a first submission. (Anything claiming 100% is cheating with the public passenger list!)

In [ ]:
import os
if os.path.exists("train.csv") and os.path.exists("test.csv"):
    train_full = add_features(pd.read_csv("train.csv"))
    test_kaggle = add_features(pd.read_csv("test.csv"))
    submit_model = make_pipeline(preprocessing, models[best_name])
    submit_model.fit(train_full[features], train_full["Survived"])
    submission = pd.DataFrame({
        "PassengerId": test_kaggle["PassengerId"],
        "Survived": submit_model.predict(test_kaggle[features]),
    })
    submission.to_csv("submission.csv", index=False)
    print("Wrote submission.csv with", len(submission), "rows")
else:
    print("Upload train.csv and test.csv from Kaggle first (see steps above).")

## ✅ What you learned
- **Classification** predicts categories; models output **probabilities** and a **threshold** turns them into yes/no.
- **Accuracy** can look good while the model is useless — check the **confusion matrix**, **precision** and **recall**.
- Clever **features** (Title, FamilySize) often matter more than fancy models.

## ✍️ Try it yourself
1. Add a feature `AgeGroup` (child / adult / senior) using `pd.cut`. Does cross-validated accuracy improve?
2. Drop the `Title` feature. How much worse does the model get?
3. Tune the random forest's `max_depth` (try 3, 5, 8, None) with cross-validation.
4. Submit to Kaggle, note your score, then try to beat it.

In [ ]:
# Your code here